In [1]:
"""Video-level time-series inference using YOLO segmentation.

Extracts fish counts and total mask surface area (in pixels) over time for
entire video recordings.
"""

from dataclasses import dataclass
import logging
import os
from pathlib import Path
import re
from typing import Dict, List, Optional, Tuple
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO

# ==============================================================================
# 1. LOGGING & WARNINGS SETUP
# ==============================================================================
warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="[%(asctime)s] [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("VideoTimeSeriesAnalysis")


# ==============================================================================
# 2. CENTRAL CONFIGURATION
# ==============================================================================
@dataclass(frozen=True)
class VideoInferenceConfig:
    """Configuration for full video inference and time-series extraction."""

    # Paths
    recordings_dir: Path = Path("./recordings_counted")
    model_path: Path = Path("./runs/segment/train-19/weights/best.pt")
    output_dir: Path = Path("./video_timeseries_output")

    # Inference hyperparameters
    conf_threshold: float = 0.25
    retina_masks: bool = True  # Ensures high-resolution pixel-accurate masks
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


CONFIG = VideoInferenceConfig()
#CONFIG.output_dir.mkdir(parents=True, exist_ok=True)

logger.info(f"Execution device: {CONFIG.device}")
logger.info(f"Recordings directory: {CONFIG.recordings_dir.resolve()}")
logger.info(f"YOLO Model path: {CONFIG.model_path.resolve()}")
logger.info("Setup and imports initialized successfully.")

[13:34:01] [INFO] Execution device: cuda
[13:34:01] [INFO] Recordings directory: /mnt/c/Users/sinke/Desktop/Masterarbeit/yolo/recordings_counted
[13:34:01] [INFO] YOLO Model path: /mnt/c/Users/sinke/Desktop/Masterarbeit/yolo/runs/segment/train-19/weights/best.pt
[13:34:01] [INFO] Setup and imports initialized successfully.


In [ ]:
gt_count = {}
gt_count['fishvideo30'] = 41
gt_count['fishvideo31'] = 15
gt_count['fishvideo32'] = 15
gt_count['fishvideo33'] = 35
gt_count['fishvideo34'] = 35
gt_count['fishvideo38'] = 44
gt_count['fishvideo39'] = 30
gt_count['fishvideo40'] = 30
gt_count['fishvideo80'] = 15
gt_count['fishvideo81'] = 15
gt_count['fishvideo82'] = 15
gt_count['fishvideo83'] = 15
gt_count['fishvideo84'] = 35
gt_count['fishvideo86'] = 30

In [4]:
# ==============================================================================
# FULL VIDEO INFERENCE & TIME-SERIES EXTRACTION
# ==============================================================================

# 1. Load YOLO model
logger.info(f"Loading YOLO model from: {CONFIG.model_path}")
model = YOLO(str(CONFIG.model_path))

# 2. Strictly filter directories matching pattern: 'frames_fishvideo{number}'
pattern = re.compile(r"^frames_fishvideo(\d+)$")

video_folders = [
    d
    for d in CONFIG.recordings_dir.iterdir()
    if d.is_dir() and pattern.match(d.name)
]

# Sort naturally by the extracted video number
video_folders = sorted(
    video_folders,
    key=lambda p: int(pattern.match(p.name).group(1)),
)

logger.info(
    f"Found {len(video_folders)} matching folder(s) with pattern 'frames_fishvideo{{number}}'."
)

# Target dictionary
results: Dict[str, List[Dict[str, int]]] = {}


# Helper function to sort frame files numerically (e.g. 1.jpg, 2.jpg, 10.jpg)
def get_sorted_frame_paths(folder: Path) -> List[Path]:
    """Finds all .jpg frame files and sorts them numerically
    """
    frame_files = [f for f in folder.glob("*.jpg") if f.stem.isdigit()]
    return sorted(frame_files, key=lambda f: int(f.stem))


# 3. Process each video frame-by-frame
for folder in tqdm(video_folders, desc="Overall Progress (Videos)"):
    # Extract clean video key name (e.g., 'frames_fishvideo30' -> 'fishvideo30')
    video_name = folder.name.replace("frames_", "")

    frame_paths = get_sorted_frame_paths(folder)
    if not frame_paths:
        logger.warning(f"No image frames found in {folder.name}. Skipping.")
        continue

    video_frame_records: List[Dict[str, int]] = []

    # Iterate through all frames of the current video
    for img_path in tqdm(
        frame_paths, desc=f"Inference: {video_name}", leave=False
    ):
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            logger.warning(f"Could not read frame: {img_path.name}")
            continue

        # Run YOLO inference
        pred = model.predict(
            source=img_bgr,
            conf=CONFIG.conf_threshold,
            retina_masks=CONFIG.retina_masks,
            verbose=False,
            device=CONFIG.device,
        )[0]

        # Extract fish count and combined mask surface area
        if pred.masks is not None and len(pred.masks) > 0:
            fish_count = len(pred.boxes) if pred.boxes is not None else 0

            # Convert boolean masks to binary array (N, H, W)
            raw_masks = (pred.masks.data.cpu().numpy() > 0.5).astype(np.uint8)

            # Combined union of all masks (avoids counting overlapping pixels multiple times)
            combined_mask = np.any(raw_masks > 0, axis=0)
            total_mask_area = int(np.sum(combined_mask))
        else:
            fish_count = 0
            total_mask_area = 0

        # Append frame entry
        video_frame_records.append(
            {
                "fish_count": fish_count,
                "mask_area": total_mask_area,
            }
        )

    # Store full frame series under the video key
    results[video_name] = video_frame_records
    logger.info(
        f"Finished {video_name}: Processed {len(video_frame_records)} frames."
    )

logger.info(
    f"Inference complete! Results dictionary populated for {len(results)} videos."
)

[13:42:43] [INFO] Loading YOLO model from: runs/segment/train-19/weights/best.pt
[13:42:43] [INFO] Found 15 matching folder(s) with pattern 'frames_fishvideo{number}'.


Overall Progress (Videos):   0%|          | 0/15 [00:00<?, ?it/s]

Inference: fishvideo30:   0%|          | 0/204 [00:00<?, ?it/s]

[13:43:21] [INFO] Finished fishvideo30: Processed 204 frames.


Inference: fishvideo31:   0%|          | 0/207 [00:00<?, ?it/s]

...

[13:50:37] [INFO] Finished fishvideo84: Processed 280 frames.


Inference: fishvideo85:   0%|          | 0/181 [00:00<?, ?it/s]

[13:51:00] [INFO] Finished fishvideo85: Processed 181 frames.


Inference: fishvideo86:   0%|          | 0/390 [00:00<?, ?it/s]

[13:52:13] [INFO] Finished fishvideo86: Processed 390 frames.
[13:52:13] [INFO] Inference complete! Results dictionary populated for 15 videos.
